In [ ]:
"""Train the MoE and save out-of-fold predictions
Nested CV: no data from the held-out outer fold (`te`) enters hyperparameter
selection, calibration, or threshold estimation at any step. All model
fitting, inner cross-validation, sigmoid/Platt calibration, and Youden
threshold estimation use only the outer training fold (`tr`); `te` is used
exclusively to generate out-of-fold predictions via .predict_proba()."""

import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from help.ckm_utils import CONDITIONS
from help.stats_utils import SEED, N_REPEATS, N_OUTER, outer_splits, entmax15, youden_threshold

In [ ]:
POP_PATH, OUT_DIR = "analytic_population.pkl", "Results_MoE"
N_INNER, N_CV = 5, 5
C_GRID = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0] 
GATE_CLASS_WEIGHT = None 
COND_NAMES, COND_COLS = list(CONDITIONS), list(CONDITIONS.values())
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_pickle(POP_PATH)
assert df["participant_id"].is_unique, "participant_id must be unique (CV is participant-independent)"
feat_cols = df.attrs["feat_cols"]
df["gender"] = df["gender"].str.lower().str.strip()

In [ ]:
def cv_split(seed):
    return StratifiedKFold(N_CV, shuffle=True, random_state=seed)

def _logit(p):                                                   
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

def fit_platt(p, y):                                             
    """Logistic recalibration p -> P(y=1), fitted on cross-validated TRAINING scores only."""
    return LogisticRegression(C=1e6, max_iter=1000).fit(_logit(p), y)

def make_expert(seed):
    pipe = Pipeline([("scale", StandardScaler()), ("pca", PCA(0.95, random_state=SEED)),
                     ("lr", LogisticRegressionCV(Cs=C_GRID, cv=cv_split(seed), scoring="roc_auc",
                                                 class_weight="balanced", max_iter=2000))])
    return CalibratedClassifierCV(pipe, method="sigmoid", cv=cv_split(seed), ensemble=False)

def fit_predict_experts(X, pos, ctrl, fit_idx, pred_idx, seed):
    """Each expert j is trained on condition-j-positive vs pure controls
    (burden_score == 0); condition-j-positive includes comorbid cases
    (burden >= 1 with other conditions present). This is intentional:
    real-world condition-positive participants are frequently multimorbid,
    so restricting training to "pure" single-condition cases would not
    reflect the population the model is meant to score."""
    out = np.zeros((len(pred_idx), pos.shape[1]))
    for j in range(pos.shape[1]):
        rows = fit_idx[pos[fit_idx, j] | ctrl[fit_idx]]
        out[:, j] = make_expert(seed).fit(X[rows], pos[rows, j].astype(int)).predict_proba(X[pred_idx])[:, 1]
    return out

In [ ]:
def run_outer_fold(X, pos, ctrl, y, tr, te, seed):
    """Nested CV: `te` (held-out outer fold) is used only to generate
    out-of-fold predictions. Every fitting: inner expert CV (i), expert
    refit (ii), z-score parameters (iii), gate CV (iv), Platt recalibration 
    (v), and Youden thresholds (vi), uses only `tr` or subsets of `tr`."""
    
    k = pos.shape[1]
    # (i) inner out-of-fold expert probabilities = the gate's training inputs
    p_tr = np.zeros((len(tr), k))
    for a, b in StratifiedKFold(N_INNER, shuffle=True, random_state=seed).split(np.zeros(len(tr)), y[tr]):
        p_tr[b] = fit_predict_experts(X, pos, ctrl, tr[a], tr[b], seed)
    # (ii) experts refit on the whole outer-training set, predict the held-out fold
    p_te = fit_predict_experts(X, pos, ctrl, tr, te, seed)
    # (iii) z-score against TRAINING controls only
    c = ctrl[tr]
    mu, sd = p_tr[c].mean(0), p_tr[c].std(0) + 1e-9
    z_tr, z_te = (p_tr - mu) / sd, (p_te - mu) / sd
    # (iv) gate: C tuned by shuffled 5-fold CV on AUC, same grid as the experts
    make_gate = lambda: LogisticRegressionCV(Cs=C_GRID, cv=cv_split(seed), scoring="roc_auc",
                                             class_weight=GATE_CLASS_WEIGHT, max_iter=2000)     
    gate = make_gate().fit(z_tr, y[tr])
    g_te_raw = gate.predict_proba(z_te)[:, 1]
    # (v) Platt recalibration on cross-validated TRAINING gate scores (C re-tuned inside each split)    
    g_tr_raw = cross_val_predict(make_gate(), z_tr, y[tr], method="predict_proba", cv=cv_split(seed))[:, 1]
    platt = fit_platt(g_tr_raw, y[tr])
    g_te = platt.predict_proba(_logit(g_te_raw))[:, 1]
    g_tr = platt.predict_proba(_logit(g_tr_raw))[:, 1]
    # (vi) thresholds from TRAINING rows only (composite on the calibrated scale)
    thr = [youden_threshold(y[tr], g_tr)]
    for j in range(k):
        pool = pos[tr, j] | c
        thr.append(youden_threshold(pos[tr][pool, j].astype(int), p_tr[pool, j]))
    flags = np.column_stack([g_te >= thr[0]] + [p_te[:, j] >= thr[1 + j] for j in range(k)])
    return (te, p_te, z_te, g_te, flags, gate.coef_[0], gate.intercept_[0], thr,
            float(gate.C_[0]), float(platt.coef_[0, 0]), float(platt.intercept_[0]))            

def run_sex(d, sex):
    d = d.reset_index(drop=True)
    X = d[feat_cols].to_numpy(float)
    pos = np.column_stack([(d[c] == 1).to_numpy() for c in COND_COLS])
    ctrl = (d["burden_score"] == 0).to_numpy()
    y = d["y_any"].to_numpy().astype(int)

    splits = list(outer_splits(y))
    out = Parallel(n_jobs=-1)(delayed(run_outer_fold)(X, pos, ctrl, y, tr, te, SEED + rep)
                              for rep, tr, te in splits)
    n, k = pos.shape
    P, Z = np.zeros((N_REPEATS, n, k)), np.zeros((N_REPEATS, n, k))
    G, F = np.zeros((N_REPEATS, n)), np.zeros((N_REPEATS, n, k + 1))
    gate_rows, thr_rows = [], []
    for i, ((rep, _, _), (te, p, z, g, fl, coef, icpt, thr, gC, ps, pi)) in enumerate(zip(splits, out)):   
        P[rep, te], Z[rep, te], G[rep, te], F[rep, te] = p, z, g, fl
        gate_rows.append({"gender": sex, "repeat": rep, "fold": i % N_OUTER, "intercept": icpt, "C": gC,
                          "platt_slope": ps, "platt_intercept": pi,                                         
                          **{f"coef_{c}": v for c, v in zip(COND_NAMES, coef)}})
        thr_rows.append({"gender": sex, "repeat": rep, "fold": i % N_OUTER,
                         **dict(zip(["composite"] + COND_NAMES, thr))})

    nk = pos.sum(1)
    shares = entmax15(Z.mean(0)) * 100
    top, rank = shares.argmax(1), (-shares).argsort(1).argsort(1)
    names = np.array(COND_NAMES, dtype=object)
    res = pd.DataFrame({"participant_id": d["participant_id"], "gender": sex,
                        "burden_score": d["burden_score"], "y_any": y,
                        "predicted_composite_score": 100 * G.mean(0),      # now 100 x calibrated probability
                        "flag_rate_composite": F[:, :, 0].mean(0)})
    for j, c in enumerate(COND_NAMES):
        res[f"true_{c}"], res[f"prob_{c}"] = pos[:, j].astype(int), P.mean(0)[:, j]
        res[f"calib_{c}"], res[f"contrib_{c}"] = Z.mean(0)[:, j], shares[:, j]
        res[f"flag_rate_{c}"] = F[:, :, 1 + j].mean(0)
    res["predicted_condition"] = names[top]
    res["is_isolated_case"] = nk == 1
    res["true_condition"] = np.where(nk == 1, names[pos.argmax(1)], None)
    res["attribution_correct"] = np.where(nk == 1, top == pos.argmax(1), np.nan)
    res["is_multimorbid"] = nk >= 2
    res["recall_at_k_score"] = np.where(nk >= 2, ((rank < nk[:, None]) & pos).sum(1) / np.maximum(nk, 1), np.nan)

    # whole-cohort Youden
    thr_all = [{"gender": sex, "scope": "composite", "threshold": youden_threshold(y, G.mean(0))}]
    for j, c in enumerate(COND_NAMES):
        pool = pos[:, j] | ctrl
        thr_all.append({"gender": sex, "scope": c,
                        "threshold": youden_threshold(pos[pool, j].astype(int), P.mean(0)[pool, j])})
    stab = [{"gender": sex, "repeat": r, "auc": roc_auc_score(y, G[r])} for r in range(N_REPEATS)]
    return res, thr_all, stab, gate_rows, thr_rows

In [ ]:
parts = [run_sex(g, sex) for sex, g in df.groupby("gender")]
oof = pd.concat([p[0] for p in parts], ignore_index=True)
oof.to_csv(f"{OUT_DIR}/oof_predictions.csv", index=False)
oof.to_pickle(f"{OUT_DIR}/moe_participant_results.pkl")
for i, name in zip([1, 2, 3, 4], ["thresholds", "cv_stability", "gate_coefficients", "fold_thresholds"]):
    pd.DataFrame([r for p in parts for r in p[i]]).to_csv(f"{OUT_DIR}/{name}.csv", index=False)
print(oof.groupby("gender").size())
print(df.groupby("gender")["y_any"].mean().round(3).rename("prevalence of y_any"))               
print(pd.DataFrame([r for p in parts for r in p[3]]).groupby("gender")[["C", "platt_slope", "platt_intercept"]].describe().T)